# Clasificación de sentimiento multilingüe — Parte 2, iteración 9 (ensamble de la iteración 6 + XLM-R-e5)

**Curso Aprendizaje de Máquina 2026-20 · Universidad de los Andes** · **Equipo en Kaggle:** Grupo 24 · **Integrantes:** _(completar)_

## Qué hace este notebook
La iteración 8 (`parte2_deep_learning_v8.ipynb`) probó dos puntos de partida nuevos con arquitectura XLM-RoBERTa base como quinta variante del ensamble de la iteración 6. El mejor candidato fue **el ensamble de la iteración 6 más `intfloat/multilingual-e5-base`**:

| Candidato (validación, 3 070 reseñas) | accuracy | frente a la iteración 6 | gana / pierde | P(ganancia > 0) |
|---|---|---|---|---|
| Ensamble de la iteración 6 (4 variantes base) | 0.9117 | — | — | — |
| **+ XLM-R-e5 + aumentación** | **0.9130** | **+0.13** | 23 / 19 (p = 0.64) | 0.71 |

No alcanzó el umbral de +0.2 puntos que se había fijado, así que la iteración 8 no generó envío. **Esta iteración lo lleva igualmente al modelo final, como excepción explícita a esa regla**:
- es la única configuración, entre todas las probadas desde la iteración 6, que quedó por encima del ensamble de la iteración 6 en validación;
- su ganancia esperada es pequeña (unas décimas) y no se distingue del azar: el *leaderboard* público puede moverse en cualquier dirección.

**Cómo.**
- Se entrena solo la variante nueva, `XLM-R-e5 + aumentación`, con el 100 % de train más todas las sintéticas, **2 épocas** (su mejor época en la validación de la iteración 8) y las semillas 42 y 7.
- Sus probabilidades de eval se promedian con las de los 8 miembros de la iteración 6, que se leen de `probabilidades_eval_v6.csv`. El promedio es primero entre semillas y después entre las 5 variantes, con pesos iguales.
- Antes de entrenar se comprueba que el promedio de los 8 miembros leídos reproduce el envío de la iteración 6.

## Cumplimiento de las reglas de la Parte 2
| Regla | Cómo se cumple |
|---|---|
| Arquitecturas profundas, solo *encoder-only* | Las 5 variantes son XLM-RoBERTa base con *fine-tuning* completo y una cabeza de clasificación propia. `intfloat/multilingual-e5-base` (licencia MIT; Wang et al., 2024) es XLM-RoBERTa base entrenado después para representar frases, y de él solo se cargan los pesos del codificador. No se usa ningún modelo generativo. |
| Aumentación de datos | El cambio de idioma con diccionarios MUSE de la iteración 6, con las mismas sintéticas, que se comprueba que coincidan. |
| Prohibido entrenar con eval | `eval.csv` solo se usa para predecir. La decisión viene de la validación de la iteración 8, que sale de train. |
| Modelo guardado | Los 2 miembros nuevos quedan en `modelo_final_v9/` y los 8 de la iteración 6 siguen en `modelo_final_v6/`. `modelo_final_v9/mezcla.json` lista los 10. |

## Cómo ejecutarlo en Google Colab (GPU L4)
1. *Entorno de ejecución → Cambiar tipo de entorno de ejecución → **GPU L4***.
2. Usa la misma carpeta de Drive. Necesita `probabilidades_eval_v6.csv` en `parte2_salida_v2` (**imprescindible**; si no está, el notebook pide subirlo, y hay copia en `parte6_salida/`). Si además encuentra `probabilidades_validacion_v8.csv`, vuelve a mostrar la evidencia de validación.
3. *Entorno de ejecución → Ejecutar todo*. Tiempo estimado en L4: **≈ 15 min**.
4. Salidas, todas copiadas a `parte2_salida_v2` en Drive:
   - `submission_parte2_v9.csv`, que se descarga sola;
   - `probabilidades_eval_v9.csv`;
   - `modelo_final_v9/` (≈ 1.1 GB);
   - `resumen_v9.json`.

**Selección en Kaggle (fijada antes de ver el puntaje).** Esta iteración y la 6 difieren solo en unas pocas reseñas de eval, así que elegir entre ellas por el público no cuesta nada en el privado (el mismo argumento de la iteración 7). Por eso:
- se selecciona **la mejor en el público entre esta y la 6**, junto con **la 5**, que es la más distinta;
- se entrega la que se seleccione de las dos;
- la iteración 1 nunca se selecciona.

## Resumen de resultados
_(se completa con las cifras de la ejecución)_

## 1. Configuración
Es la misma de las iteraciones 6 a 8. `VARIANTES_BASE` son las 4 variantes de la iteración 6, que no se reentrenan. `NUEVAS` es la variante que entra, con su mejor época en la validación de la iteración 8.

In [ ]:
import os, sys, re, gc, json, time, math, copy, random, shutil, unicodedata, collections, urllib.request
from pathlib import Path

EN_COLAB = 'google.colab' in sys.modules
SMOKE_TEST = os.environ.get('SMOKE_TEST') == '1'      # prueba local rápida: CPU, subconjunto y modelos diminutos
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import transformers
from transformers import (AutoTokenizer, AutoModel, AutoConfig, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)
import sklearn
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
transformers.logging.set_verbosity_error()

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USA_AMP = DEVICE.type == 'cuda'                       # precisión mixta solo en GPU
GPU_NOMBRE, GPU_GB, BF16_NATIVO = 'cpu', 0.0, False
if DEVICE.type == 'cuda':
    _p = torch.cuda.get_device_properties(0)
    GPU_NOMBRE, GPU_GB = _p.name, _p.total_memory / 2**30
    BF16_NATIVO = torch.cuda.get_device_capability(0)[0] >= 8      # Ampere / Ada (A100, L4); la T4 es 7.5
    torch.backends.cuda.matmul.allow_tf32 = True                  # TF32 en las operaciones que quedan en FP32
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
DTYPE_AMP = torch.bfloat16 if BF16_NATIVO else torch.float16
USA_SCALER = USA_AMP and DTYPE_AMP == torch.float16   # BF16 no necesita escalar la pérdida
GPU_GRANDE = GPU_GB >= 20                             # L4 (24 GB) o A100: lotes de inferencia más grandes
NUM_WORKERS = min(4, os.cpu_count() or 2) if EN_COLAB else 0
T_INICIO = time.time()
print(f'torch {torch.__version__} | transformers {transformers.__version__} | scikit-learn {sklearn.__version__} '
      f'| pandas {pd.__version__} | dispositivo: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {GPU_NOMBRE} | {GPU_GB:.1f} GB | precisión mixta: {str(DTYPE_AMP).replace("torch.", "")} '
          f'| GradScaler: {USA_SCALER} | CPU para cargar datos: {NUM_WORKERS} procesos')
elif not SMOKE_TEST:
    print('⚠️ Sin GPU: Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU L4')

# Estilo de las gráficas: superficie y tinta neutras, paleta categórica en orden fijo y rampa azul secuencial
SUPERFICIE, TINTA, TINTA2, TENUE, REJILLA, EJE = '#fcfcfb', '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7'
SERIES = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100']
RAMPA_AZUL = LinearSegmentedColormap.from_list('azul', ['#f4f8fd', '#cde2fb', '#86b6ef', '#3987e5', '#256abf', '#184f95', '#0d366b'])
plt.rcParams.update({
    'figure.facecolor': SUPERFICIE, 'axes.facecolor': SUPERFICIE, 'savefig.facecolor': SUPERFICIE,
    'axes.edgecolor': EJE, 'axes.labelcolor': TINTA2, 'xtick.color': TENUE, 'ytick.color': TINTA2,
    'text.color': TINTA, 'axes.spines.top': False, 'axes.spines.right': False,
    'grid.color': REJILLA, 'grid.linestyle': '-', 'grid.linewidth': 0.8,
    'axes.titlelocation': 'left', 'axes.titlesize': 12, 'font.size': 10, 'font.family': 'DejaVu Sans',
})

In [ ]:
SEED = 42
CLASES = ['negativo', 'neutral', 'positivo']
C2I = {c: i for i, c in enumerate(CLASES)}
IDIOMAS = ['es', 'en', 'pt', 'fr', 'es+en']

MODELO = 'xlm-roberta-base'                                       # XLM-R original (MIT)
MODELO_SENT = 'clapAI/roberta-base-multilingual-sentiment'        # ajustado a sentimiento (Apache-2.0)
MODELO_E5 = 'intfloat/multilingual-e5-base'                       # XLM-R base entrenado para representar frases (MIT)

# Las 4 variantes de la iteración 6: no se reentrenan (se leen sus probabilidades de eval)
VARIANTES_BASE = {
    'XLM-R sin aumentación':     (MODELO, 'sin aumentación', 4),
    'XLM-R + aumentación':       (MODELO, 'aumentación estática', 3),
    'XLM-R-sent + aumentación':  (MODELO_SENT, 'aumentación estática', 3),
    'XLM-R-sent + consistencia': (MODELO_SENT, 'consistencia', 2),
}
# La variante nueva elegida en la validación de la iteración 8 (mejor época: 2)
NUEVAS = {'XLM-R-e5 + aumentación': (MODELO_E5, 'aumentación estática')}
MEJOR_NUEVA = {'XLM-R-e5 + aumentación': {'época': 2}}
NUEVAS_ELEGIDAS = list(NUEVAS)
ELEGIDO = 'Ensamble de 5 variantes (base + XLM-R-e5 + aumentación)'
REFERENCIA = 'Ensamble de 4 variantes base (iteración 5)'   # el ensamble de la iteración 6 (nombre en sus archivos)
SEMILLAS_ANTERIORES = [42, 7]
SEMILLAS_FINAL = [42, 7]
ITERACION_ANTERIOR = '_v6'
PERFILES = {m: dict(BATCH=32, LR_BACKBONE=2.5e-5, LR_CABEZA=1e-4, LLRD=0.9) for m in (MODELO, MODELO_SENT, MODELO_E5)}
MAX_LEN = 256
LAMBDA_CONS = 1.0
EPOCAS = 4
PERDIDA_COLAPSO = 1.0
BATCH_PRED = 256 if GPU_GRANDE else 128
WARMUP = 0.1
WEIGHT_DECAY = 0.01
DROPOUT = 0.1
SUAVIZADO = 0.05

# Aumentación: la misma de la iteración 6
URL_DICCIONARIO = 'https://dl.fbaipublicfiles.com/arrival/dictionaries/{}-{}.txt'   # MUSE (Conneau et al., 2018), CC BY-NC 4.0
MEZCLAS = [('es', 'en', 6000),   # (idioma de la reseña, idioma que entra, cuántas reseñas; None = todas las de ese idioma)
           ('pt', 'en', None), ('fr', 'en', None),
           ('en', 'es', None), ('en', 'pt', None), ('en', 'fr', None)]
PROP_CAMBIO = (0.3, 0.6)         # fracción de las palabras con traducción que cambian de idioma en cada reseña
LARGO_TRAMO = (1, 4)             # cambian en tramos de 1 a 4 palabras seguidas, como en las mezclas reales
P_PRIMERA = 0.7                  # prob. de usar la primera traducción del diccionario; si no, otra de las 3 primeras

ENTRENAR_FINAL = True
GUARDAR_EN_DRIVE = True
URL_CARPETA_DRIVE = 'https://drive.google.com/drive/folders/157vOjCwRql_YtsS-Leby3oOQ_U9LYw5t'
RUTA_DRIVE_MANUAL = None         # solo si no se encuentra la carpeta: su ruta montada, p. ej. '/content/drive/MyDrive/proyecto ml'
NOMBRE_SALIDA = 'parte2_salida_v2'   # la misma subcarpeta de salidas de las iteraciones 2 a 8
SUFIJO = '_v9'                   # los archivos de esta iteración llevan este sufijo, así no pisan los de las anteriores
DESCARGAR_MODELO = False
GUARDAR_BF16 = True

if SMOKE_TEST:
    MEJOR_NUEVA = {v: {'época': 1} for v in MEJOR_NUEVA}
    for _perfil in PERFILES.values():
        _perfil['BATCH'] = 16


def carpeta_drive(url, raiz=Path('/content/drive'), profundidad=4, max_carpetas=3000):
    """Ruta montada de la carpeta de Drive de un enlace (Colab monta Drive por rutas, no por ID)."""
    id_carpeta = re.search(r'folders/([\w-]+)', url).group(1)
    # 1) acceso directo a una carpeta compartida: /content/drive/.shortcut-targets-by-id/<ID>/<nombre>
    atajo = raiz / '.shortcut-targets-by-id' / id_carpeta
    if atajo.is_dir():
        hijas = [p for p in atajo.iterdir() if p.is_dir()]
        if len(hijas) == 1:
            return hijas[0]
    # 2) recorrido de Mi unidad comparando el atributo extendido user.drive.id de cada carpeta
    if hasattr(os, 'getxattr'):
        pendientes, vistas = collections.deque([(raiz / 'MyDrive', 0)]), 0
        while pendientes and vistas < max_carpetas:
            carpeta, nivel = pendientes.popleft()
            vistas += 1
            try:
                if os.getxattr(carpeta, 'user.drive.id').decode() == id_carpeta:
                    return carpeta
            except OSError:
                pass
            if nivel < profundidad:
                try:
                    pendientes.extend((h, nivel + 1) for h in sorted(carpeta.iterdir())
                                      if not h.name.startswith('.') and h.is_dir())
                except OSError:
                    pass
    # 3) API de Drive: nombre y carpetas madre hasta la raíz de Mi unidad
    try:
        from google.colab import auth
        from googleapiclient.discovery import build
        auth.authenticate_user()
        servicio = build('drive', 'v3', cache_discovery=False)
        id_raiz = servicio.files().get(fileId='root', fields='id').execute()['id']
        nombres, actual = [], id_carpeta
        while actual != id_raiz:
            meta = servicio.files().get(fileId=actual, fields='name,parents', supportsAllDrives=True).execute()
            if not meta.get('parents'):           # fuera de Mi unidad (p. ej. solo en «Compartido conmigo»)
                return None
            nombres.append(meta['name'])
            actual = meta['parents'][0]
        ruta = raiz / 'MyDrive' / Path(*reversed(nombres))
        return ruta if ruta.is_dir() else None
    except Exception as e:                        # sin permisos o sin la API: se informa abajo
        print('No se pudo consultar la API de Drive:', e)
        return None


DIR_DRIVE = None
DIR_SALIDA_DRIVE = None
if GUARDAR_EN_DRIVE and EN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DIR_DRIVE = Path(RUTA_DRIVE_MANUAL) if RUTA_DRIVE_MANUAL else carpeta_drive(URL_CARPETA_DRIVE)
    if DIR_DRIVE is None or not DIR_DRIVE.is_dir():
        raise FileNotFoundError(
            'No encuentro la carpeta de Drive del enlace. Si es de otra persona: en drive.google.com, clic derecho → '
            'Organizar → Añadir acceso directo → Mi unidad, y vuelve a ejecutar esta celda. '
            'También puedes escribir su ruta en RUTA_DRIVE_MANUAL (panel de archivos → drive → clic derecho → Copiar ruta).')
    print('Carpeta del proyecto en Drive:', DIR_DRIVE)
    DIR_SALIDA_DRIVE = DIR_DRIVE / NOMBRE_SALIDA
    DIR_SALIDA = Path(f'/content/{NOMBRE_SALIDA}') # Guardar localmente para evitar IO lenta
else:
    DIR_SALIDA = Path(os.environ.get('PARTE2_SALIDA', f'/content/{NOMBRE_SALIDA}' if EN_COLAB else NOMBRE_SALIDA))
DIR_SALIDA.mkdir(parents=True, exist_ok=True)


def fijar_semilla(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def liberar():
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()


fijar_semilla(SEED)
print('Salidas locales temporales en', DIR_SALIDA.resolve())
if DIR_SALIDA_DRIVE is not None:
    print('Al finalizar, todo se copiará automáticamente a', DIR_SALIDA_DRIVE.resolve())

## 2. Datos
Los mismos datos y la misma heurística de idioma, necesaria para regenerar las sintéticas. `eval.csv` solo se usa en la sección 5, para predecir.

In [ ]:
def buscar_datos():
    candidatos = [Path(os.environ['PARTE2_DATOS'])] if os.environ.get('PARTE2_DATOS') else []
    if DIR_DRIVE is not None:                         # carpeta del proyecto en Drive y sus subcarpetas (p. ej. data/)
        candidatos += [DIR_DRIVE] + sorted(p for p in DIR_DRIVE.iterdir() if p.is_dir() and p.name != NOMBRE_SALIDA)
    candidatos += [Path('/content'), Path('/content/data'), Path('/content/sample_data'), Path('data'), Path('.')]
    for c in candidatos:
        if (c / 'train.csv').exists() and (c / 'eval.csv').exists() and (c / 'sample_submission.csv').exists():
            return c
    return None


DIR_DATOS = buscar_datos()
if DIR_DATOS is None and EN_COLAB:
    from google.colab import files
    print('No encuentro los CSV: sube train.csv, eval.csv y sample_submission.csv')
    files.upload()                                    # quedan en /content
    DIR_DATOS = buscar_datos()
assert DIR_DATOS is not None, 'Faltan train.csv, eval.csv o sample_submission.csv'

train = pd.read_csv(DIR_DATOS / 'train.csv')
evaluacion = pd.read_csv(DIR_DATOS / 'eval.csv')
muestra = pd.read_csv(DIR_DATOS / 'sample_submission.csv')
assert {'id', 'text', 'label'} <= set(train.columns) and {'id', 'text'} <= set(evaluacion.columns)
assert list(muestra.columns) == ['id', 'answer'] and (evaluacion.id.values == muestra.id.values).all()
assert set(train.label) == set(CLASES) and train.text.notna().all() and evaluacion.text.notna().all()
if SMOKE_TEST:
    train = train.sample(600, random_state=SEED).reset_index(drop=True)
    evaluacion, muestra = evaluacion.head(40).copy(), muestra.head(40).copy()
print(f'Datos en {DIR_DATOS} | train {train.shape} | eval {evaluacion.shape} | textos duplicados en train: '
      f'{train.text.duplicated().sum()}')
display(train.label.value_counts().to_frame('reseñas').assign(proporción=lambda d: (d.reseñas / d.reseñas.sum()).round(3)))

In [ ]:
PALABRAS_IDIOMA = {
    'es': set('el los las y pero muy del al una lo mi me es por con mas tambien compre llego fue esta este esto ya hace'.split()),
    'en': set('the and is was with this that it my for but very not have had are just really i im its so of to in on at after they'.split()),
    'pt': set('nao muito com em do da na os ao pra isso ele ela foi tem uma um meu minha voce tambem ja'.split()),
    'fr': set('le les et est une pour avec tres je mon pas qui du au ce cette il elle ai sur dans ete j l d c n qu'.split()),
}


def _sin_tildes(s):
    return ''.join(c for c in unicodedata.normalize('NFKD', str(s).lower()) if not unicodedata.combining(c))


def detectar_idioma(texto):
    palabras = re.findall(r'[a-z]+', _sin_tildes(texto))
    n = max(1, len(palabras))
    s = {k: sum(p in v for p in palabras) / n for k, v in PALABRAS_IDIOMA.items()}
    if s['es'] >= 0.06 and s['en'] >= 0.06 and min(s['es'], s['en']) >= 0.3 * max(s['es'], s['en']):
        return 'es+en'
    return max(s, key=s.get)


train['idioma'] = train.text.map(detectar_idioma)
tabla_idioma = pd.crosstab(train.idioma, train.label).reindex(IDIOMAS).fillna(0).astype(int)
tabla_idioma['total'] = tabla_idioma.sum(axis=1)
tabla_idioma['% de train'] = (100 * tabla_idioma.total / len(train)).round(1)
print('Reseñas de train por idioma (heurística) y etiqueta:')
display(tabla_idioma)

TOKENIZADORES = {}


def tokenizador(nombre):
    """Tokenizador de cada transformer (se carga una sola vez)."""
    if nombre not in TOKENIZADORES:
        TOKENIZADORES[nombre] = AutoTokenizer.from_pretrained(nombre)
    return TOKENIZADORES[nombre]

## 3. Aumentación: las mismas sintéticas de la iteración 6
Se regeneran con la misma semilla y se comparan con `aumentacion_diccionarios.csv` en Drive.

In [ ]:
def _clave(palabra):
    """Minúsculas y sin tildes, pero conservando la ñ (año ≠ ano, uñas ≠ unas)."""
    return _sin_tildes(palabra.replace('ñ', '\0').replace('Ñ', '\0')).replace('\0', 'ñ')


def cargar_diccionario(origen, destino):
    """Diccionario bilingüe de MUSE → (por palabra exacta en minúsculas, por palabra sin tildes), con las
    traducciones en el orden del archivo."""
    ruta = DIR_SALIDA / 'diccionarios' / f'{origen}-{destino}.txt'
    if not ruta.exists():
        ruta.parent.mkdir(parents=True, exist_ok=True)
        copia = DIR_SALIDA_DRIVE / 'diccionarios' / ruta.name if DIR_SALIDA_DRIVE is not None else None
        if copia is not None and copia.exists():          # descargado en una ejecución anterior
            shutil.copy(copia, ruta)
        else:
            urllib.request.urlretrieve(URL_DICCIONARIO.format(origen, destino), ruta)
    exacto, sin_tildes = collections.defaultdict(list), collections.defaultdict(list)
    for linea in ruta.read_text(encoding='utf-8').splitlines():
        partes = linea.split()
        if len(partes) == 2 and _clave(partes[0]) != _clave(partes[1]):
            for dic, clave in [(exacto, partes[0].lower()), (sin_tildes, _clave(partes[0]))]:
                if partes[1] not in dic[clave]:
                    dic[clave].append(partes[1])
    return dict(exacto), dict(sin_tildes)


def traducciones(palabra, diccionario):
    """Primero la palabra exacta. Si no está y tiene 5 letras o más, sin tildes (≈30 % de las reseñas no las lleva);
    en palabras cortas eso confundiría pares como te / té o el / él."""
    exacto, sin_tildes = diccionario
    p = palabra.lower()
    if p in exacto:
        return exacto[p]
    return sin_tildes.get(_clave(p)) if len(p) >= 5 else None


RE_TOKEN = re.compile(r'\w+|[^\w\s]+|\s+')        # palabras, signos y espacios: unidos reconstruyen el texto exacto


def mezclar_idiomas(texto, dic, rng):
    """Cambia de idioma tramos de palabras con el diccionario. None si hay menos de 3 palabras traducibles."""
    tokens = RE_TOKEN.findall(str(texto))
    posiciones = [i for i, t in enumerate(tokens) if t[0].isalpha() and traducciones(t, dic)]
    if len(posiciones) < 3:
        return None
    objetivo = rng.uniform(*PROP_CAMBIO) * len(posiciones)
    cambiar = set()
    for k in rng.permutation(len(posiciones)):
        if len(cambiar) >= objetivo:
            break
        cambiar.update(posiciones[k:k + int(rng.integers(LARGO_TRAMO[0], LARGO_TRAMO[1] + 1))])
    for i in cambiar:
        opciones = traducciones(tokens[i], dic)[:3]
        nueva = opciones[0] if len(opciones) == 1 or rng.random() < P_PRIMERA else opciones[int(rng.integers(1, len(opciones)))]
        tokens[i] = nueva[:1].upper() + nueva[1:] if tokens[i][:1].isupper() else nueva
    return ''.join(tokens)

In [ ]:
COLUMNAS_AUG = ['text', 'label', 'origen_id', 'tipo', 'idioma']
t0 = time.time()
rng = np.random.default_rng(SEED)
filas, DICCIONARIOS = [], {}
for origen, destino, n in MEZCLAS:
    if (origen, destino) not in DICCIONARIOS:
        DICCIONARIOS[(origen, destino)] = cargar_diccionario(origen, destino)
    dic = DICCIONARIOS[(origen, destino)]
    fuente = train[train.idioma == origen]
    if n is not None and n < len(fuente):
        fuente = fuente.sample(n, random_state=SEED)
    antes = len(filas)
    for i, texto, etiqueta in fuente[['id', 'text', 'label']].itertuples(index=False):
        nuevo = mezclar_idiomas(texto, dic, rng)
        if nuevo is not None and nuevo != texto:
            filas.append((nuevo, etiqueta, i, f'{origen}→{origen}+{destino}', f'{origen}+{destino}'))
    print(f'  {origen}→{origen}+{destino}: diccionario {origen}-{destino} con {len(dic[0]):,} palabras | '
          f'{len(filas) - antes:,} sintéticas de {len(fuente):,} reseñas')
AUG = pd.DataFrame(filas, columns=COLUMNAS_AUG).drop_duplicates('text').reset_index(drop=True)
print(f'Aumentación generada: {len(AUG):,} reseñas sintéticas en {time.time() - t0:.0f} s')

# Comprobación: los miembros nuevos deben entrenarse con las mismas sintéticas que los de la iteración 6
AUG_IGUAL_ANTERIOR = None
copia = DIR_SALIDA_DRIVE / 'aumentacion_diccionarios.csv' if DIR_SALIDA_DRIVE is not None else None
if copia is not None and copia.exists():
    anterior = pd.read_csv(copia, keep_default_na=False)
    AUG_IGUAL_ANTERIOR = bool(len(anterior) == len(AUG) and (anterior.text.values == AUG.text.values).all()
                              and (anterior.label.values == AUG.label.values).all())
    print('Iguales a las sintéticas de la iteración 6 (aumentacion_diccionarios.csv en Drive):', AUG_IGUAL_ANTERIOR)
    if not AUG_IGUAL_ANTERIOR:
        print('⚠️ No coinciden: se usan las de la iteración 6, que son con las que se entrenaron sus miembros')
        AUG = anterior[COLUMNAS_AUG].copy()
else:
    print('No está la copia de la iteración 6 en Drive para compararla; se usan las recién generadas')
display(pd.crosstab(AUG.tipo, AUG.label, margins=True))

## 4. Definiciones del modelo, miembros de la iteración 6 y evidencia de validación
Las definiciones son idénticas a las de la iteración 6. Después se leen las probabilidades de eval de los 8 miembros de la iteración 6 y se comprueba que su promedio reproduce su envío. Si está en Drive, también se lee la validación de la iteración 8, para mostrar de nuevo la evidencia que motivó esta iteración.

In [ ]:
class DatasetTokens(Dataset):
    def __init__(self, textos, tok, y=None):
        self.ids = tok(list(textos), truncation=True, max_length=MAX_LEN)['input_ids']
        self.y = y

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        return self.ids[i], (-1 if self.y is None else int(self.y[i]))


class Relleno:
    """Arma cada lote rellenando hasta su reseña más larga con el token de relleno del tokenizador del modelo."""
    def __init__(self, id_relleno):
        self.id_relleno = id_relleno

    def __call__(self, lote):
        largo = max(len(s) for s, _ in lote)
        ids = torch.full((len(lote), largo), self.id_relleno, dtype=torch.long)
        mascara = torch.zeros((len(lote), largo), dtype=torch.long)
        for j, (s, _) in enumerate(lote):
            ids[j, :len(s)] = torch.tensor(s)
            mascara[j, :len(s)] = 1
        return ids, mascara, torch.tensor([y for _, y in lote])


IDIOMA_DESTINO = {'es': 'en', 'es+en': 'en', 'pt': 'en', 'fr': 'en', 'en': ('es', 'pt', 'fr')}


class DatasetPares(Dataset):
    """Cada reseña junto con una versión mezclada NUEVA en cada época (cambio de idioma dinámico, como en CoSDA-ML)."""
    def __init__(self, textos, idiomas, y, tok, semilla):
        self.textos, self.idiomas, self.y, self.tok, self.semilla = list(textos), list(idiomas), y, tok, semilla
        self.ids = tok(self.textos, truncation=True, max_length=MAX_LEN)['input_ids']
        self.epoca = 0

    def __len__(self):
        return len(self.textos)

    def __getitem__(self, i):
        rng = np.random.default_rng([self.semilla, self.epoca, i])        # distinta en cada época y reproducible
        origen = 'es' if self.idiomas[i] == 'es+en' else self.idiomas[i]
        destinos = IDIOMA_DESTINO.get(self.idiomas[i], 'en')
        destino = destinos if isinstance(destinos, str) else destinos[int(rng.integers(len(destinos)))]
        dic = DICCIONARIOS.get((origen, destino))
        mezclado = mezclar_idiomas(self.textos[i], dic, rng) if dic is not None else None
        ids_mezcla = self.ids[i] if mezclado is None else self.tok(mezclado, truncation=True, max_length=MAX_LEN)['input_ids']
        return self.ids[i], ids_mezcla, int(self.y[i])


class RellenoPares:
    """Lote de pares: (reseña, máscara, versión mezclada, máscara, etiqueta)."""
    def __init__(self, id_relleno):
        self.relleno = Relleno(id_relleno)

    def __call__(self, lote):
        ids, mascara, y = self.relleno([(o, e) for o, _, e in lote])
        ids_m, mascara_m, _ = self.relleno([(m, e) for _, m, e in lote])
        return ids, mascara, ids_m, mascara_m, y


def perdida_consistencia(lo, lm, y):
    """Entropía cruzada en la reseña y en su versión mezclada + KL simétrica entre las dos predicciones (xTune, R-Drop)."""
    ce = 0.5 * (F.cross_entropy(lo, y, label_smoothing=SUAVIZADO) + F.cross_entropy(lm, y, label_smoothing=SUAVIZADO))
    lpo, lpm = F.log_softmax(lo, dim=1), F.log_softmax(lm, dim=1)
    kl = 0.5 * (F.kl_div(lpm, lpo, log_target=True, reduction='batchmean') +
                F.kl_div(lpo, lpm, log_target=True, reduction='batchmean'))
    return ce + LAMBDA_CONS * kl


def crear_backbone(nombre):
    if SMOKE_TEST:                                    # transformer diminuto aleatorio: solo prueba el código
        cfg = AutoConfig.from_pretrained(nombre)
        cfg.hidden_size, cfg.num_hidden_layers, cfg.num_attention_heads, cfg.intermediate_size = 32, 2, 2, 64
        return AutoModel.from_config(cfg)
    # float32 explícito: transformers 5 carga cada modelo en el tipo de su checkpoint, y el de mDeBERTa está en float16.
    # Con pesos en float16, AdamW no puede actualizarlos (los pasos son menores que la precisión) y el modelo no aprende.
    return AutoModel.from_pretrained(nombre, dtype=torch.float32)


class ClasificadorTransformer(nn.Module):
    def __init__(self, nombre=MODELO, dropout=DROPOUT, backbone=None, tok=None):
        super().__init__()
        self.nombre = nombre
        self.tok = tok if tok is not None else tokenizador(nombre)
        self.backbone = backbone if backbone is not None else crear_backbone(nombre)
        h = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.cabeza = nn.Linear(2 * h, len(CLASES))

    def forward(self, input_ids, attention_mask):
        oculto = self.backbone(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        m = attention_mask.unsqueeze(-1).to(oculto.dtype)
        media = (oculto * m).sum(1) / m.sum(1).clamp(min=1.0)
        return self.cabeza(self.dropout(torch.cat([oculto[:, 0], media], dim=1)))


def optimizador_llrd(modelo, perfil):
    """AdamW con tasa por capa: LR_BACKBONE · LLRD^(capas por encima). Embeddings la más baja; cabeza LR_CABEZA."""
    n = modelo.backbone.config.num_hidden_layers
    sin_decay = ('bias', 'LayerNorm.weight', 'layer_norm.weight')
    grupos = collections.defaultdict(list)
    for nombre, p in modelo.named_parameters():
        if nombre.startswith('backbone.'):
            capa = re.search(r'encoder\.layer\.(\d+)\.', nombre)
            if capa:
                profundidad = int(capa.group(1)) + 1
            elif 'embeddings' in nombre or 'encoder.LayerNorm' in nombre:   # incluye las posiciones relativas de DeBERTa
                profundidad = 0
            else:
                profundidad = n
            lr = perfil['LR_BACKBONE'] * perfil['LLRD'] ** (n - profundidad)
        else:
            lr = perfil['LR_CABEZA']
        grupos[(lr, 0.0 if any(s in nombre for s in sin_decay) else WEIGHT_DECAY)].append(p)
    return torch.optim.AdamW([{'params': ps, 'lr': lr, 'weight_decay': wd} for (lr, wd), ps in grupos.items()],
                             fused=DEVICE.type == 'cuda')


@torch.no_grad()
def predecir_transformer(modelo, textos, batch=None):
    modelo.eval()
    dl = DataLoader(DatasetTokens(textos, modelo.tok), batch_size=batch or BATCH_PRED,
                    collate_fn=Relleno(modelo.tok.pad_token_id), num_workers=NUM_WORKERS)
    salida = []
    for ids, mascara, _ in dl:
        with torch.autocast(device_type=DEVICE.type, dtype=DTYPE_AMP, enabled=USA_AMP):
            logits = modelo(ids.to(DEVICE), mascara.to(DEVICE))
        salida.append(torch.softmax(logits.float(), dim=1).cpu().numpy())
    return np.concatenate(salida)


@torch.no_grad()
def predecir_clasificador_publicado(nombre, textos):
    """Probabilidades del clasificador tal como se publicó (con su propia cabeza), sin ajustarlo a nuestros datos."""
    tok = tokenizador(nombre)
    clf = AutoModelForSequenceClassification.from_pretrained(nombre, dtype=torch.float32).to(DEVICE).eval()
    etiquetas = [clf.config.id2label[i].lower() for i in range(clf.config.num_labels)]
    orden = [etiquetas.index(c) for c in ('negative', 'neutral', 'positive')]       # al orden de CLASES
    dl = DataLoader(DatasetTokens(textos, tok), batch_size=BATCH_PRED, collate_fn=Relleno(tok.pad_token_id),
                    num_workers=NUM_WORKERS)
    salida = []
    for ids, mascara, _ in dl:
        with torch.autocast(device_type=DEVICE.type, dtype=DTYPE_AMP, enabled=USA_AMP):
            logits = clf(input_ids=ids.to(DEVICE), attention_mask=mascara.to(DEVICE)).logits
        salida.append(torch.softmax(logits.float(), dim=1)[:, orden].cpu().numpy())
    del clf
    liberar()
    return np.concatenate(salida)


def entrenar_transformer(textos, y, textos_val=None, y_val=None, epocas=EPOCAS, semilla=SEED, etiqueta='', nombre=MODELO,
                         idiomas=None, reintentar=True):
    """Fine-tuning completo desde el punto de partida `nombre`. Con validación: evalúa cada época y devuelve el modelo de
    la mejor. Con `idiomas`: entrenamiento con consistencia (cada reseña con una versión mezclada nueva en cada época).
    Si tras la época 1 la pérdida sigue en el nivel del azar, se reinicia una vez con otra semilla."""
    perfil = PERFILES[nombre]
    consistencia = idiomas is not None
    fijar_semilla(semilla)
    modelo = ClasificadorTransformer(nombre).to(DEVICE)
    if consistencia:
        datos, colacion = DatasetPares(textos, idiomas, y, modelo.tok, semilla), RellenoPares(modelo.tok.pad_token_id)
    else:
        datos, colacion = DatasetTokens(textos, modelo.tok, y), Relleno(modelo.tok.pad_token_id)
    dl = DataLoader(datos, batch_size=perfil['BATCH'], shuffle=True, collate_fn=colacion, num_workers=NUM_WORKERS,
                    pin_memory=DEVICE.type == 'cuda')
    opt = optimizador_llrd(modelo, perfil)
    total = len(dl) * epocas
    sched = get_linear_schedule_with_warmup(opt, int(WARMUP * total), total)
    scaler = torch.amp.GradScaler('cuda', enabled=USA_SCALER)      # solo con FP16 (T4); con BF16 no hace falta
    historial, mejor = [], {'acc': -1.0, 'época': epocas, 'semilla': semilla}
    extra = ' (+ una versión mezclada nueva de cada una por época)' if consistencia else ''
    print(f'[{etiqueta}] {len(textos):,} reseñas{extra} | {len(dl)} pasos por época | {epocas} épocas')
    for ep in range(1, epocas + 1):
        if consistencia:
            datos.epoca = ep                       # los procesos del DataLoader se crean en cada época y ven el valor nuevo
        modelo.train(); t_ep = time.time(); suma, n = 0.0, 0
        for paso, lote in enumerate(dl, 1):
            lote = [x.to(DEVICE, non_blocking=True) for x in lote]
            yb = lote[-1]
            with torch.autocast(device_type=DEVICE.type, dtype=DTYPE_AMP, enabled=USA_AMP):
                logits = modelo(lote[0], lote[1])
                logits_m = modelo(lote[2], lote[3]) if consistencia else None
            if consistencia:
                loss = perdida_consistencia(logits.float(), logits_m.float(), yb)
            else:
                loss = F.cross_entropy(logits.float(), yb, label_smoothing=SUAVIZADO)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            suma += loss.item() * len(yb); n += len(yb)
            if paso % 250 == 0:
                print(f'   época {ep} paso {paso}/{len(dl)} pérdida {suma / n:.4f} ({time.time() - t_ep:.0f} s)')
        fila = {'época': ep, 'pérdida train': suma / n, 'minutos': (time.time() - t_ep) / 60}
        if ep == 1 and reintentar and not SMOKE_TEST and fila['pérdida train'] > PERDIDA_COLAPSO:
            print(f"[{etiqueta}] ⚠️ pérdida {fila['pérdida train']:.3f} tras la época 1 (el azar es ≈ 1.10): el entrenamiento "
                  f"no despegó; se reinicia una vez con la semilla {semilla + 1000}")
            del modelo, opt, sched, dl, datos, lote, logits, logits_m, loss
            liberar()
            return entrenar_transformer(textos, y, textos_val, y_val, epocas, semilla + 1000, etiqueta, nombre, idiomas,
                                        reintentar=False)
        if textos_val is not None:
            P = predecir_transformer(modelo, textos_val)
            fila['acc val'] = accuracy_score(y_val, P.argmax(1))
            if fila['acc val'] > mejor['acc']:
                mejor = {'acc': fila['acc val'], 'época': ep, 'P': P, 'semilla': semilla,
                         'estado': {k: v.detach().cpu().clone() for k, v in modelo.state_dict().items()}}
        historial.append(fila)
        print(f'[{etiqueta}] época {ep}: ' + ' | '.join(f'{k} {v:.4f}' for k, v in fila.items() if k != 'época'))
    if textos_val is not None:
        modelo.load_state_dict(mejor.pop('estado'))
    return modelo, mejor, pd.DataFrame(historial)


def guardar_modelo(modelo, carpeta, extra=None):
    carpeta.mkdir(parents=True, exist_ok=True)
    estado = {k: (v.to(torch.bfloat16) if GUARDAR_BF16 and v.is_floating_point() else v)
              for k, v in modelo.backbone.state_dict().items()}
    modelo.backbone.save_pretrained(carpeta / 'backbone', state_dict=estado)
    modelo.tok.save_pretrained(carpeta / 'backbone')
    torch.save(modelo.cabeza.state_dict(), carpeta / 'cabeza.pt')
    (carpeta / 'config_clasificador.json').write_text(json.dumps(
        {'modelo_base': modelo.nombre, 'max_len': MAX_LEN, 'representacion': 'concat(token inicial, media)',
         'clases': CLASES, 'dropout': DROPOUT, 'pesos_backbone': 'bfloat16' if GUARDAR_BF16 else 'float32',
         **(extra or {})}, indent=1, ensure_ascii=False))


def cargar_modelo(carpeta):
    """Reconstruye un clasificador guardado: backbone y tokenizador de transformers + cabeza lineal."""
    cfg = json.loads((carpeta / 'config_clasificador.json').read_text())
    modelo = ClasificadorTransformer(cfg['modelo_base'], dropout=cfg['dropout'],
                                     backbone=AutoModel.from_pretrained(carpeta / 'backbone', dtype=torch.float32),
                                     tok=AutoTokenizer.from_pretrained(carpeta / 'backbone'))
    modelo.cabeza.load_state_dict(torch.load(carpeta / 'cabeza.pt', map_location='cpu'))
    return modelo.to(DEVICE).eval()


def promedio(por_variante):
    """Promedia las probabilidades primero entre semillas de cada variante y después entre variantes (pesos iguales)."""
    return np.mean([np.mean(v, axis=0) for v in por_variante.values()], axis=0)


def carpeta_de(variante, semilla):
    """Nombre de carpeta de un miembro: 'XLM-R + aumentación', 42 → 'xlm_r_con_aumentacion_semilla_42'."""
    return re.sub(r'[^a-z0-9]+', '_', _sin_tildes(variante.replace('+', ' con '))).strip('_') + f'_semilla_{semilla}'

In [ ]:
def buscar_anterior(nombre):
    """Archivo de la iteración 6: en la carpeta de salidas de Drive, en /content o junto a los datos."""
    for c in [DIR_SALIDA_DRIVE, DIR_SALIDA, Path('/content'), Path('/content/data'), DIR_DATOS, DIR_DRIVE]:
        if c is not None and (Path(c) / nombre).exists():
            return Path(c) / nombre
    return None


def leer_anterior(nombre):
    ruta = buscar_anterior(nombre)
    if ruta is None and EN_COLAB:
        from google.colab import files
        print(f'No encuentro {nombre} en Drive: súbelo (hay una copia en parte6_salida/)')
        files.upload()                                # queda en /content
        ruta = buscar_anterior(nombre)
    assert ruta is not None, f'Falta {nombre} (salida de la iteración 6)'
    return ruta


# Eval de la iteración 6: los 8 miembros (4 variantes × 2 semillas)
VARIANTES = VARIANTES_BASE                           # nombre que usan las líneas siguientes (iteración 7)
NOMBRE_PROB_ANT = f'probabilidades_eval{ITERACION_ANTERIOR}.csv'
RUTA_PROB_ANT = leer_anterior(NOMBRE_PROB_ANT)

prob_ant = pd.read_csv(RUTA_PROB_ANT).set_index('id').reindex(evaluacion.id)
assert prob_ant.notna().all().all(), f'{NOMBRE_PROB_ANT} no tiene todas las reseñas de eval'
P_ANTERIORES = {v: [prob_ant[[f'{v} · semilla {s} | {c}' for c in CLASES]].to_numpy() for s in SEMILLAS_ANTERIORES]
                for v in VARIANTES}
P_ANT = promedio(P_ANTERIORES)                       # el promedio de la iteración 6, recalculado desde sus miembros
DIF_ANT = float(np.abs(P_ANT - prob_ant[CLASES].to_numpy()).max())
print(f'{RUTA_PROB_ANT}: {len(VARIANTES) * len(SEMILLAS_ANTERIORES)} miembros leídos | el promedio recalculado difiere '
      f'del guardado como mucho en {DIF_ANT:.1e} (las probabilidades se guardaron con 5 decimales)')
assert DIF_ANT < 1e-3
COINCIDE_ENVIO_ANT = None
ruta_sub_ant = RUTA_PROB_ANT.parent / f'submission_parte2{ITERACION_ANTERIOR}.csv'
if ruta_sub_ant.exists():
    envio_ant = pd.read_csv(ruta_sub_ant).set_index('id').reindex(evaluacion.id).answer.to_numpy()
    COINCIDE_ENVIO_ANT = float((envio_ant == np.array(CLASES)[P_ANT.argmax(1)]).mean())
    print(f'Reproduce el envío de la iteración 6 en el {100 * COINCIDE_ENVIO_ANT:.2f} % de las reseñas de eval')
    assert COINCIDE_ENVIO_ANT > 0.999

# Modelos guardados de la iteración 6 (solo para el modelo guardado; el envío no depende de ellos)
DIR_MODELO_ANT = (DIR_SALIDA_DRIVE or DIR_SALIDA) / f'modelo_final{ITERACION_ANTERIOR}'
SEMILLA_EFECTIVA = {}                                # distinta de la nominal solo si un entrenamiento se reinició
if (DIR_MODELO_ANT / 'mezcla.json').exists():
    for m in json.loads((DIR_MODELO_ANT / 'mezcla.json').read_text())['miembros']:
        SEMILLA_EFECTIVA[(m['variante'], m['semilla'])] = m.get('semilla_efectiva', m['semilla'])
FALTAN_ANT = [carpeta_de(v, s) for v in VARIANTES for s in SEMILLAS_ANTERIORES if not (DIR_MODELO_ANT / carpeta_de(v, s)).is_dir()]
if FALTAN_ANT:
    print(f'⚠️ No encuentro en {DIR_MODELO_ANT} los miembros {FALTAN_ANT}. El envío no cambia, pero el modelo guardado '
          'completo necesita esas carpetas de la iteración 6.')
else:
    print(f'Los {len(VARIANTES) * len(SEMILLAS_ANTERIORES)} miembros de la iteración 6 están en {DIR_MODELO_ANT}')

# Evidencia de validación de la iteración 8 (si su archivo está en Drive): referencia frente al candidato elegido
EVIDENCIA = None
ruta_val_v8 = buscar_anterior('probabilidades_validacion_v8.csv')
if ruta_val_v8 is None:
    print('No encuentro probabilidades_validacion_v8.csv: la evidencia está en el notebook de la iteración 8 (sección 6).')
else:
    va8 = pd.read_csv(ruta_val_v8)
    y8 = va8.label.map(C2I).to_numpy()
    P_r = va8[[f'{REFERENCIA} | {c}' for c in CLASES]].to_numpy()
    P_c = va8[[f'{ELEGIDO} | {c}' for c in CLASES]].to_numpy()
    ok_r, ok_c = P_r.argmax(1) == y8, P_c.argmax(1) == y8
    EVIDENCIA = {'reseñas': len(va8), 'accuracy_referencia': round(float(ok_r.mean()), 4),
                 'accuracy_candidato': round(float(ok_c.mean()), 4),
                 'gana': int((ok_c & ~ok_r).sum()), 'pierde': int((~ok_c & ok_r).sum())}
    print(f'Validación de la iteración 8 ({len(va8):,} reseñas): referencia {EVIDENCIA["accuracy_referencia"]:.4f} | '
          f'{ELEGIDO} {EVIDENCIA["accuracy_candidato"]:.4f} '
          f'({100 * (EVIDENCIA["accuracy_candidato"] - EVIDENCIA["accuracy_referencia"]):+.2f} pts; '
          f'gana {EVIDENCIA["gana"]}, pierde {EVIDENCIA["pierde"]})')

## 5. Modelo final: XLM-R-e5 con 2 semillas, ensamble y envío
`XLM-R-e5 + aumentación` se entrena con **las 20 466 reseñas de train más todas las sintéticas**, 2 épocas y las semillas 42 y 7. Sus probabilidades de eval se promedian con las de los 8 miembros de la iteración 6: primero entre semillas, después entre las 5 variantes. **Aquí, y solo aquí, se usa `eval.csv`, únicamente para predecir.** Al final se cuenta cuántas predicciones cambian frente al envío de la iteración 6.

In [ ]:
DIR_MODELO = DIR_SALIDA / f'modelo_final{SUFIJO}'
ENVIO = ENTRENAR_FINAL and bool(NUEVAS_ELEGIDAS)
MIEMBROS_NUEVOS = [(v, NUEVAS[v][0], NUEVAS[v][1], MEJOR_NUEVA[v]['época'], s) for v in NUEVAS_ELEGIDAS for s in SEMILLAS_FINAL]
MINUTOS_FINAL, MINUTOS_MIEMBRO = 0.0, {}
if ENVIO:
    X_base, y_base = train.text.to_numpy(), train.label.map(C2I).to_numpy()
    X_aug_full = np.concatenate([X_base, AUG.text.to_numpy()])
    y_aug_full = np.concatenate([y_base, AUG.label.map(C2I).to_numpy()])
    textos_eval = evaluacion.text.to_numpy()
    t0 = time.time()
    P_NUEVOS = collections.defaultdict(list)
    for k, (variante, nombre, entrenamiento, ep, s) in enumerate(MIEMBROS_NUEVOS):
        t_m = time.time()
        etiqueta = f'final {k + 1}/{len(MIEMBROS_NUEVOS)}: {variante}, semilla {s}'
        modelo_final, info, _ = entrenar_transformer(X_aug_full, y_aug_full, epocas=ep, semilla=s, etiqueta=etiqueta, nombre=nombre)
        SEMILLA_EFECTIVA[(variante, s)] = info['semilla']
        P_NUEVOS[variante].append(predecir_transformer(modelo_final, textos_eval))
        CARPETA_ULTIMO = DIR_MODELO / carpeta_de(variante, s)
        guardar_modelo(modelo_final, CARPETA_ULTIMO, {'variante': variante, 'semilla': info['semilla'], 'epocas': int(ep),
                                                      'entrenamiento': entrenamiento})
        MINUTOS_MIEMBRO[etiqueta] = round((time.time() - t_m) / 60, 1)
        if k < len(MIEMBROS_NUEVOS) - 1:
            del modelo_final
            liberar()
    # Ensamble final: las 4 variantes de la iteración 6 (2 semillas cada una) + las nuevas (2 semillas cada una)
    P_EVAL_MIEMBROS = {**{v: P_ANTERIORES[v] for v in VARIANTES_BASE}, **dict(P_NUEVOS)}
    P_eval = promedio(P_EVAL_MIEMBROS)
    MIEMBROS_TODOS = ([(v, *VARIANTES_BASE[v], s) for v in VARIANTES_BASE for s in SEMILLAS_ANTERIORES] + MIEMBROS_NUEVOS)

    def carpeta_relativa(variante, semilla):
        iteracion = ITERACION_ANTERIOR if variante in VARIANTES_BASE else SUFIJO
        return f'modelo_final{iteracion}/{carpeta_de(variante, semilla)}'

    (DIR_MODELO / 'mezcla.json').write_text(json.dumps(
        {'configuracion': ELEGIDO,
         'miembros': [{'variante': v, 'carpeta': carpeta_relativa(v, s), 'punto_de_partida': ini, 'entrenamiento': ent,
                       'epocas': int(ep), 'semilla': s, 'semilla_efectiva': SEMILLA_EFECTIVA.get((v, s), s)}
                      for v, ini, ent, ep, s in MIEMBROS_TODOS],
         'carpetas_relativas_a': NOMBRE_SALIDA, 'lambda_consistencia': LAMBDA_CONS,
         'promedio': 'primero entre semillas de cada variante y después entre variantes (pesos iguales)',
         'peso_tfidf': 0.0, 'aumentacion_tipo': 'cambio de idioma con diccionarios MUSE',
         'pesos_backbone': 'bfloat16' if GUARDAR_BF16 else 'float32'},
        ensure_ascii=False, indent=1))
    MINUTOS_FINAL = (time.time() - t0) / 60
    print(f'{len(MIEMBROS_NUEVOS)} miembros nuevos entrenados y eval predicho en {MINUTOS_FINAL:.1f} min | ensamble final: '
          f'{len(MIEMBROS_TODOS)} miembros ({len(P_EVAL_MIEMBROS)} variantes × 2 semillas)')
else:
    print('No hay modelo final nuevo: se mantiene el de la iteración 6 (submission_parte2_v6.csv).')

In [ ]:
CAMBIOS = None
if ENVIO:
    submission = pd.DataFrame({'id': evaluacion.id, 'answer': np.array(CLASES)[P_eval.argmax(1)]})
    assert list(submission.columns) == list(muestra.columns) == ['id', 'answer']
    assert len(submission) == len(muestra) and (submission.id.values == muestra.id.values).all()
    assert submission.answer.isin(CLASES).all()
    RUTA_SUB = DIR_SALIDA / f'submission_parte2{SUFIJO}.csv'
    submission.to_csv(RUTA_SUB, index=False)
    print(f'{RUTA_SUB} generado: {submission.shape}')
    display(pd.DataFrame({'eval (predicho)': submission.answer.value_counts(normalize=True),
                          'train (real)': train.label.value_counts(normalize=True)}).round(3))

    recargado = cargar_modelo(CARPETA_ULTIMO)
    P_a = predecir_transformer(modelo_final, textos_eval[:64])
    P_b = predecir_transformer(recargado, textos_eval[:64])
    print(f'Modelo recargado ({CARPETA_ULTIMO.name}): coinciden {int((P_a.argmax(1) == P_b.argmax(1)).sum())} de {len(P_a)} '
          f'predicciones | diferencia máxima de probabilidad {np.abs(P_a - P_b).max():.1e}')
    del recargado
    liberar()

    # Cuánto cambia frente al envío de la iteración 6 (solo se cuentan predicciones; eval no tiene etiquetas)
    cambia = P_eval.argmax(1) != P_ANT.argmax(1)
    conf = P_eval.max(1)
    CAMBIOS = {'reseñas': int(cambia.sum()), 'con_confianza_menor_0.7': int((cambia & (conf < 0.7)).sum())}
    print(f'\nFrente al envío de la iteración 6 cambian {CAMBIOS["reseñas"]} reseñas de eval '
          f'({100 * cambia.mean():.1f} %), {CAMBIOS["con_confianza_menor_0.7"]} de ellas con confianza < 0.7')
    if cambia.any():
        display(pd.crosstab(pd.Series(np.array(CLASES)[P_ANT.argmax(1)][cambia], name='iteración 6'),
                            pd.Series(np.array(CLASES)[P_eval.argmax(1)][cambia], name='esta iteración')))

    prob_eval = pd.DataFrame({'id': evaluacion.id, **{c: P_eval[:, j].round(5) for j, c in enumerate(CLASES)}})
    for variante, lista in P_EVAL_MIEMBROS.items():
        semillas_v = SEMILLAS_ANTERIORES if variante in VARIANTES_BASE else SEMILLAS_FINAL
        for s, P_s in zip(semillas_v, lista):
            for j, c in enumerate(CLASES):
                prob_eval[f'{variante} · semilla {s} | {c}'] = P_s[:, j].round(5)
    prob_eval.to_csv(DIR_SALIDA / f'probabilidades_eval{SUFIJO}.csv', index=False)
    if EN_COLAB:
        from google.colab import files
        files.download(str(RUTA_SUB))
        if DESCARGAR_MODELO:
            files.download(shutil.make_archive(str(DIR_MODELO), 'zip', DIR_MODELO))
print('Archivos en', DIR_SALIDA.resolve(), ':', sorted(p.name for p in DIR_SALIDA.iterdir()))

**Cómo cargar el modelo en otra sesión.** Hay que ejecutar las secciones 1, 2 y la de definiciones de la sección 4. `modelo_final_v9/mezcla.json` lista los 10 miembros con su carpeta relativa a `parte2_salida_v2`: 8 en `modelo_final_v6/` y 2 en `modelo_final_v9/`.
```python
miembros = json.loads((DIR_SALIDA_DRIVE / 'modelo_final_v9' / 'mezcla.json').read_text())['miembros']
P = np.mean([predecir_transformer(cargar_modelo(DIR_SALIDA_DRIVE / m['carpeta']), ['texto de una reseña'])
             for m in miembros], axis=0)
```
Las 5 variantes tienen 2 semillas cada una, así que ese promedio simple coincide con el de la sección 5.

## 6. Resumen numérico y copia de las salidas a Drive

In [ ]:
RESUMEN = {
    'versiones': {'torch': torch.__version__, 'transformers': transformers.__version__, 'sklearn': sklearn.__version__},
    'gpu': {'nombre': GPU_NOMBRE, 'gb': round(GPU_GB, 1), 'precision': str(DTYPE_AMP), 'grad_scaler': USA_SCALER},
    'smoke_test': SMOKE_TEST,
    'config': {'variantes_base': {v: {'punto_de_partida': ini, 'entrenamiento': ent, 'epocas': ep}
                                  for v, (ini, ent, ep) in VARIANTES_BASE.items()},
               'nuevas': {v: {'punto_de_partida': ini, 'entrenamiento': ent, 'epocas': MEJOR_NUEVA[v]['época']}
                          for v, (ini, ent) in NUEVAS.items()},
               'perfiles': PERFILES, 'semillas_final': SEMILLAS_FINAL, 'max_len': MAX_LEN},
    'comprobaciones': {'dif_max_eval_anterior': DIF_ANT, 'coincide_envio_anterior': COINCIDE_ENVIO_ANT,
                       'sinteticas_iguales_a_la_anterior': AUG_IGUAL_ANTERIOR, 'faltan_modelos_anteriores': FALTAN_ANT},
    'evidencia_validacion_iteracion_8': EVIDENCIA,
    'final': {'configuracion': ELEGIDO, 'envio': ENVIO,
              'miembros_nuevos': [{'variante': v, 'punto_de_partida': ini, 'epocas': int(ep), 'semilla': s,
                                   'semilla_efectiva': SEMILLA_EFECTIVA.get((v, s))} for v, ini, _, ep, s in MIEMBROS_NUEVOS],
              'minutos': round(MINUTOS_FINAL, 1), 'minutos_por_miembro': MINUTOS_MIEMBRO,
              'cambios_frente_a_la_iteracion_6': CAMBIOS,
              'distribucion_eval': submission.answer.value_counts().to_dict() if ENVIO else None},
    'salidas': str(DIR_SALIDA_DRIVE or DIR_SALIDA),
    'minutos_totales': round((time.time() - T_INICIO) / 60, 1),
}
print(json.dumps(RESUMEN, ensure_ascii=False, indent=1, default=str))
(DIR_SALIDA / f'resumen{SUFIJO}.json').write_text(json.dumps(RESUMEN, ensure_ascii=False, indent=1, default=str), encoding='utf-8')

if DIR_DRIVE is not None and DIR_SALIDA_DRIVE is not None:
    print(f'\nCopiando archivos de {DIR_SALIDA} a Drive ({DIR_SALIDA_DRIVE})...')
    import shutil
    from google.colab import drive
    if not DIR_DRIVE.is_dir():                 # Drive se desconectó durante la ejecución: se vuelve a montar
        drive.mount('/content/drive', force_remount=True)
    shutil.copytree(DIR_SALIDA, DIR_SALIDA_DRIVE, dirs_exist_ok=True)
    drive.flush_and_unmount()
    print('Copia completada. Drive sincronizado y desmontado. Salidas en', DIR_SALIDA_DRIVE)

## Conclusiones
_(se completa con las cifras de la ejecución)_